# 📘 Kit 1 — Collecte de données via APIs
## Formation Data Engineer

---

### Comment utiliser ce notebook ?

Chaque bloc de code est clairement identifié :

- **`# ✅ CODE FOURNI`** → ce code vous est donné, lisez-le attentivement et exécutez-le
- **`# 🔧 TODO`** → vous devez compléter ce bloc vous-mêmes

Les cellules Markdown (comme celle-ci) expliquent le contexte et les notions. **Lisez-les avant d'exécuter le code.**

### Prérequis
- Avoir `communes_isere.csv` dans le même dossier que ce notebook
- Python 3.8+

### Livrables attendus
- `images_filtered.json`
- `communes_enrichies.csv`
- `entreprises_raw.csv`
- `dataset_final_kit1.csv`
- `rapport_qualite.json`

---
## Section 0 — Installation et configuration

Exécutez cette cellule **une seule fois** pour installer les bibliothèques nécessaires.

In [4]:
# ✅ CODE FOURNI — Installation des bibliothèques
%pip install requests pandas --quiet
print('✓ Installation terminée')

Note: you may need to restart the kernel to use updated packages.
✓ Installation terminée


In [5]:
# ✅ CODE FOURNI — Imports et configuration
import requests   # Pour faire des requêtes HTTP vers les APIs
import json       # Pour lire/écrire des fichiers JSON
import csv        # Pour lire/écrire des fichiers CSV
import time       # Pour ajouter des pauses entre les requêtes
import os         # Pour créer des dossiers
import pandas as pd  # Pour nettoyer et fusionner les données
from collections import Counter  # Pour compter les occurrences

os.makedirs('data', exist_ok=True)
print(f'✓ Imports OK — pandas {pd.__version__}, requests {requests.__version__}')

✓ Imports OK — pandas 3.0.1, requests 2.32.5


---
## Section 1 — La fonction safe_get() : votre outil de base

### Pourquoi une fonction spéciale pour les requêtes ?

En théorie, faire une requête API c'est simple : `requests.get(url)`. En pratique, plein de choses peuvent mal se passer :
- Le serveur met trop de temps à répondre → votre script reste bloqué indéfiniment
- L'API retourne une erreur (404, 429, 500) → requests ne lève pas d'exception automatiquement
- Un problème réseau → une exception Python non gérée fait planter tout votre script

La fonction `safe_get()` encapsule tous ces cas. **Vous l'utiliserez dans TOUT ce notebook.** Ne faites jamais `requests.get()` directement dans vos exercices.

### Anatomie de la fonction

In [6]:
# ✅ CODE FOURNI — Fonction safe_get() à utiliser dans tout le notebook

def safe_get(url, params=None):
    """
    Effectue une requête GET sécurisée vers l'URL donnée.
    
    Paramètres :
        url    : l'adresse de l'API (ex: 'https://geo.api.gouv.fr/communes')
        params : dictionnaire de paramètres optionnels (ex: {'nom': 'Grenoble'})
    
    Retourne : les données JSON sous forme de dict/liste Python, ou None en cas d'erreur.
    """
    try:
        # timeout=10 : abandonne si l'API ne répond pas en 10 secondes
        # Sans timeout, votre script peut rester bloqué indéfiniment !
        r = requests.get(url, params=params, timeout=10)
        
        # raise_for_status() transforme les codes 4xx et 5xx en exception Python
        # Sans ça, une erreur 404 passerait silencieusement
        r.raise_for_status()
        
        # .json() décode le corps de la réponse en dict ou liste Python
        return r.json()
    
    except requests.exceptions.Timeout:
        # L'API n'a pas répondu en 10 secondes
        print(f'⏱ Timeout : {url}')
        return None
    
    except requests.exceptions.HTTPError as e:
        # L'API a répondu avec un code d'erreur (400, 404, 429, 500...)
        print(f'❌ Erreur HTTP {e.response.status_code} : {url}')
        return None
    
    except Exception as e:
        # Toute autre erreur (problème réseau, JSON malformé...)
        print(f'❌ Erreur inattendue : {e}')
        return None

# Test rapide pour vérifier que la connexion fonctionne
test = safe_get('https://picsum.photos/v2/list', {'limit': 1})
if test:
    print('✓ safe_get() fonctionne correctement')
    print(f'  Exemple de réponse : {test[0]}')
else:
    print('❌ Problème de connexion — vérifiez votre réseau')

✓ safe_get() fonctionne correctement
  Exemple de réponse : {'id': '0', 'author': 'Alejandro Escamilla', 'width': 5000, 'height': 3333, 'url': 'https://unsplash.com/photos/yC-Yzbqy7PY', 'download_url': 'https://picsum.photos/id/0/5000/3333'}


---
## Section 2 — Partie 1 : API Picsum

### Contexte

L'API Picsum (https://picsum.photos) fournit une liste d'images libres de droits. Elle ne nécessite pas d'authentification — c'est donc parfaite pour s'exercer avant d'attaquer les APIs gouvernementales.

**Endpoint principal** : `https://picsum.photos/v2/list`

**Paramètres utiles** :
- `page` : numéro de la page (commence à 1)
- `limit` : nombre de résultats par page (max 100)

### Exercice 2.1 — Explorer la structure de l'API

**Rappel de la question du kit :** Faites une première requête GET sur l'API Picsum. Quel est le code de statut retourné ? Que contient la réponse ?

In [7]:
# ✅ CODE FOURNI — Première requête et exploration de la structure
# Objectif : comprendre ce que retourne l'API avant d'automatiser quoi que ce soit

# On fait d'abord la requête 'à la main' (sans safe_get) pour voir le code de statut
response = requests.get('https://picsum.photos/v2/list', timeout=10)

print(f'Code de statut : {response.status_code}')  # 200 = OK
print(f'Type de la réponse brute : {type(response)}')
print(f'Que contient response.text[:100] : {response.text[:100]}')

Code de statut : 200
Type de la réponse brute : <class 'requests.models.Response'>
Que contient response.text[:100] : [{"id":"0","author":"Alejandro Escamilla","width":5000,"height":3333,"url":"https://unsplash.com/pho


In [10]:
# 🔧 TODO — Q4 & Q5 : Explorer la structure de la réponse
# 
# Q4. Appelez .json() sur la réponse pour obtenir les données.
#     Affichez le type de l'objet retourné et le nombre d'éléments.
#     Quel code de statut avez-vous reçu ? Que signifie-t-il ?
# 
# Q5. Affichez le premier élément complet.
#     Listez les clés disponibles. À quoi correspond le champ 'author' ?
data = response.json()

# TODO : complétez les quatre lignes suivantes
print(f'Type : {type(data)}')
print(f"Nombre d'éléments : {len(data)}")
print(f'Premier élément :{data[0]}')
print(f'Clés disponibles : {data[0].keys()}')


Type : <class 'list'>
Nombre d'éléments : 30
Premier élément :{'id': '0', 'author': 'Alejandro Escamilla', 'width': 5000, 'height': 3333, 'url': 'https://unsplash.com/photos/yC-Yzbqy7PY', 'download_url': 'https://picsum.photos/id/0/5000/3333'}
Clés disponibles : dict_keys(['id', 'author', 'width', 'height', 'url', 'download_url'])


### Exercice 2.2 — Pagination et paramètres

L'API retourne 30 images par défaut. Pour en récupérer 100, il faut utiliser les **paramètres** `page` et `limit` :

```
https://picsum.photos/v2/list?page=1&limit=100
```

En Python avec `requests`, on passe les paramètres dans un dictionnaire — la bibliothèque construit automatiquement l'URL :

```python
safe_get(url, {'page': 1, 'limit': 100})
```

**Question du kit :** En passant les paramètres `page` et `limit`, récupérez exactement 100 images.

In [14]:
# 🔧 TODO — Q6 : Récupérer 100 images via les paramètres
#
# Utilisez safe_get() avec les paramètres de pagination.
# Vérifiez que vous avez bien 100 images.

images = safe_get('https://picsum.photos/v2/list', {'page': 1, 'limit': 100})  # TODO : paramètres

if images:
    print(f'✓ {len(images)} images récupérées')  # Doit afficher 100
    print(f"Exemples d'auteurs : {[img['author'] for img in images[:5]]}")
else:
    print('❌ Requête échouée')


✓ 100 images récupérées
Exemples d'auteurs : ['Alejandro Escamilla', 'Alejandro Escamilla', 'Alejandro Escamilla', 'Alejandro Escamilla', 'Alejandro Escamilla']


### Exercice 2.3 — Filtrage et export JSON

**Question du kit :** Filtrez les images dont le prénom de l'auteur contient la lettre 'r' (insensible à la casse). Sauvegardez dans `images_filtered.json`.

**Aide sur le filtrage :**
- `img['author']` donne le nom complet, ex : `'Alejandro Escamilla'`
- `.split()` découpe en mots : `['Alejandro', 'Escamilla']`
- `[0]` prend le premier mot (le prénom) : `'Alejandro'`
- `.lower()` met en minuscules : `'alejandro'`
- `'r' in 'alejandro'` → `False` (pas de 'r')

Voici comment déboguer pour comprendre ce que split() produit :

In [15]:
# ✅ CODE FOURNI — Debug : voir ce que split() produit sur les auteurs
# Exécutez cette cellule pour comprendre comment fonctionne le filtrage

print('Démonstration du filtrage :')
for img in images[:5]:  # Afficher les 5 premiers auteurs
    prenom = img['author'].split()[0]          # Premier mot = prénom
    prenom_lower = prenom.lower()              # En minuscules
    contient_r = 'r' in prenom_lower           # Test
    print(f'  {img["author"]:25s} → prénom="{prenom}" → lower="{prenom_lower}" → contient "r" : {contient_r}')

Démonstration du filtrage :
  Alejandro Escamilla       → prénom="Alejandro" → lower="alejandro" → contient "r" : True
  Alejandro Escamilla       → prénom="Alejandro" → lower="alejandro" → contient "r" : True
  Alejandro Escamilla       → prénom="Alejandro" → lower="alejandro" → contient "r" : True
  Alejandro Escamilla       → prénom="Alejandro" → lower="alejandro" → contient "r" : True
  Alejandro Escamilla       → prénom="Alejandro" → lower="alejandro" → contient "r" : True


In [20]:
# 🔧 TODO — Q7 & Q8 : Filtrage et export JSON
#
# Q7. Créez la liste 'filtered' en filtrant les images dont le prénom
#     de l'auteur contient la lettre 'r' (insensible à la casse).
#     Combien en obtenez-vous ?
#
# Q8. Sauvegardez cette liste dans images_filtered.json.

filtered = [img for img in images if 'r' in img['author'].lower()]  # TODO : condition de filtrage

print(f'{len(filtered)} images filtrées sur {len(images)}')

# TODO : sauvegardez filtered dans images_filtered.json
with open('images_fltered.json', 'w', encoding='utf-8') as f:
          json.dump(filtered, f, ensure_ascii=False, indent=4)
print('✓ images_filtered.json créé')


80 images filtrées sur 100
✓ images_filtered.json créé


---
## Section 3 — Partie 2.1 : API Communes

### Contexte

Vous avez reçu `communes_isere.csv` — un fichier avec 49 communes de l'Isère.
Votre mission : l'enrichir avec les coordonnées GPS et le code postal de chaque commune.

**API :** `https://geo.api.gouv.fr/communes`  
**Paramètres :** `nom`, `fields`, `limit`, **`codeDepartement`**

> ⚠️ **Pourquoi `codeDepartement=38` ?**  
> Sans ce paramètre, des communes homonymes d'autres départements remontent en premier.  
> Par exemple, "Vienne" existe dans l'Isère (38) ET dans la Vienne (86).  
> En filtrant sur `codeDepartement=38`, on garantit de toujours obtenir la commune iséroise.

> ⚠️ **Format GeoJSON :** les coordonnées sont retournées dans l'ordre `[longitude, latitude]` — l'inverse de l'usage courant.


In [ ]:
# 🔧 TODO — Q9 : Explorer manuellement l'API Communes
#
# Avant d'automatiser, testez toujours l'API manuellement.
# Faites une requête pour la commune de Grenoble avec les champs
# nom, codesPostaux et centre. Affichez la réponse complète.
# Répondez en commentaire : dans quel ordre sont les coordonnées ?

result = safe_get('https://geo.api.gouv.fr/communes', {
    "nom": "Grenoble",
    "fields": "nom,codesPostaux,centre",
    "limit": 1,
    "codeDepartement": "38"
})


# Affichez le résultat de manière lisible
print(json.dumps(result, indent=2, ensure_ascii=False))

# TODO : répondez ici
# coords[0] = ???
# coords[1] = ???


[
  {
    "nom": "Grenoble",
    "codesPostaux": [
      "38000",
      "38100"
    ],
    "centre": {
      "type": "Point",
      "coordinates": [
        5.7155,
        45.1842
      ]
    },
    "code": "38185",
    "_score": 1
  }
]
https://geo.api.gouv.fr/communes?nom=Grenoble&codeDepartement=38&fields=centre%2CcodesPostaux&limit=1
[{'centre': {'type': 'Point', 'coordinates': [5.7155, 45.1842]}, 'codesPostaux': ['38000', '38100'], 'nom': 'Grenoble', 'code': '38185', '_score': 1}]


In [42]:
# ✅ CODE FOURNI — Fonction d'enrichissement : filtre département + correspondance exacte
# Deux filtres :
#   1. codeDepartement=38 → uniquement l'Isère
#   2. Correspondance exacte (avec gestion des accents et des articles)
#      Exemples : 'Saint-Egrève' == 'Saint-Égrève'  /  'Pont-de-Claix' == 'Le Pont-de-Claix'

import unicodedata

def normaliser(s):
    """Minuscules + suppression des accents pour une comparaison robuste."""
    s = s.lower()
    s = unicodedata.normalize('NFD', s)   # décompose les caractères accentués
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')  # supprime les diacritiques

def enrich_commune(nom):
    data = safe_get('https://geo.api.gouv.fr/communes', {
        'nom':             nom,
        'fields':          'nom,codesPostaux,centre',
        'codeDepartement': '38',
        'limit':           10
    })
    base = {'nom_commune': nom, 'code_postal': None, 'latitude': None, 'longitude': None}
    if data is None:     return {**base, 'statut': 'API_ERROR'}
    if len(data) == 0:   return {**base, 'statut': 'NOT_FOUND'}

    # Correspondance exacte avec normalisation des accents
    # et prise en charge des articles définis (Le/La/Les)
    nom_norm = normaliser(nom)
    exact = [
        r for r in data
        if normaliser(r['nom']) == nom_norm
        or normaliser(r['nom']) in ('le ' + nom_norm, 'la ' + nom_norm, 'les ' + nom_norm)
    ]

    if len(exact) == 0:   return {**base, 'statut': 'NOT_FOUND'}

    c      = exact[0]
    codes  = c.get('codesPostaux', [])
    coords = c.get('centre', {}).get('coordinates', [None, None])
    return {
        'nom_commune': nom,
        'code_postal': codes[0] if codes else None,
        'latitude':  coords[1],
        'longitude': coords[0],
        'statut': 'OK'
    }

# Vérification des cas qui posaient problème
for t in ['Pont-de-Claix', 'Saint-Egrève', 'Bourg-d\'Oisans', 'Claix', 'Moirans', 'Vienne']:
    r = enrich_commune(t)
    print(f'{r["nom_commune"]:25s} → statut={r["statut"]:5s}  lat={r["latitude"]}')


Pont-de-Claix             → statut=OK     lat=45.123
Saint-Egrève              → statut=OK     lat=45.2315
Bourg-d'Oisans            → statut=OK     lat=45.0367
Claix                     → statut=OK     lat=45.1187
Moirans                   → statut=OK     lat=45.3227
Vienne                    → statut=OK     lat=45.5221


In [ ]:
# 🔧 TODO — Q10, Q11 & Q12 : Enrichir toutes les communes
#
# Q10. Lisez communes_isere.csv et appelez enrich_commune() pour chaque ligne.
#      Ajoutez time.sleep(0.1) entre chaque appel.
#      Pourquoi faut-il cette pause ?
#
# Q11. Certaines communes retournent plusieurs résultats ou sont introuvables.
#      Affichez les communes dont le statut n'est pas OK.
#
# Q12. Exportez dans communes_enrichies.csv.
#      Combien ont le statut OK ? Lesquelles posent problème ?

if not os.path.exists('communes_isere.csv'):
    print('❌ communes_isere.csv introuvable')
else:
    results = []

    # TODO : lire communes_isere.csv et enrichir chaque commune
    with open('communes_isere.csv', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            commune_enrichie = enrich_commune(row["nom_commune"])  # TODO
            results.append(commune_enrichie)
            time.sleep(0.1)

    # TODO : affichez le comptage des statuts (Counter)
    comptage = Counter(r['statut'] for r in results)
    print(f'✓ {len(results)} communes traitées — Statuts : {dict(comptage)}')

    for r in results:
        if r['statut'] != 'OK':
            print(f'  {str(r["nom_commune"]):30s} → {r["statut"]}')

    # TODO : exporter dans communes_enrichies.csv
    fieldnames = ['nom_commune', 'code_postal', 'latitude', 'longitude', 'statut']
    with open('communes_enrichies.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results) # TODO
    print('✓ communes_enrichies.csv créé')

✓ 49 communes traitées — Statuts : {'NOT_FOUND': 49}
  {'nom': 'Grenoble'}            → NOT_FOUND
  {'nom': 'Voiron'}              → NOT_FOUND
  {'nom': 'Vienne'}              → NOT_FOUND
  {'nom': 'Bourgoin-Jallieu'}    → NOT_FOUND
  {'nom': 'Échirolles'}          → NOT_FOUND
  {'nom': "Saint-Martin-d'Hères"} → NOT_FOUND
  {'nom': 'Villefontaine'}       → NOT_FOUND
  {'nom': 'Crolles'}             → NOT_FOUND
  {'nom': 'Meylan'}              → NOT_FOUND
  {'nom': 'Pont-de-Claix'}       → NOT_FOUND
  {'nom': 'Vizille'}             → NOT_FOUND
  {'nom': 'Voreppe'}             → NOT_FOUND
  {'nom': 'Sassenage'}           → NOT_FOUND
  {'nom': 'Fontaine'}            → NOT_FOUND
  {'nom': 'Saint-Egrève'}        → NOT_FOUND
  {'nom': 'Rives'}               → NOT_FOUND
  {'nom': 'Roussillon'}          → NOT_FOUND
  {'nom': 'Tullins'}             → NOT_FOUND
  {'nom': 'La Tour-du-Pin'}      → NOT_FOUND
  {'nom': 'Moirans'}             → NOT_FOUND
  {'nom': 'Saint-Marcellin'}     → NOT_FOUND
 

### Vérification du résultat

Avec `codeDepartement=38` + correspondance exacte normalisée, vous devriez obtenir **49 communes en statut OK**.

La fonction `normaliser()` gère deux cas :
- **Accents** : `'Saint-Egrève'` reconnaît `'Saint-Égrève'` dans l'API
- **Articles** : `'Pont-de-Claix'` reconnaît `'Le Pont-de-Claix'` dans l'API


In [ ]:
# ✅ CODE FOURNI — Analyse et nettoyage du résultat d'enrichissement

from collections import Counter

# Comptage des statuts
comptage = Counter(r['statut'] for r in results)
print(f'✓ {len(results)} communes traitées')
print(f'Statuts : {dict(comptage)}')
print()

# Afficher les cas problématiques
pb = [r for r in results if r['statut'] != 'OK']
if pb:
    print(f'Cas à vérifier ({len(pb)}) :')
    for r in pb:
        print(f'  {r["nom_commune"]:30s} → {r["statut"]}')
else:
    print('✓ Toutes les communes ont été trouvées avec statut OK')

print()

# Statistiques finales
avec_coords = sum(1 for r in results if r['latitude'] is not None)
print(f'Communes avec coordonnées GPS : {avec_coords} / {len(results)}')

# Exporter quand même toutes les communes (y compris NOT_FOUND)
# Les NOT_FOUND seront ignorées lors de la collecte des entreprises
print()
print('Note : les communes NOT_FOUND seront ignorées lors de la collecte des entreprises.')
print('       Elles n\'ont pas de coordonnées GPS → impossible de faire la recherche géographique.')


---
## Section 4 — Partie 2.2 : API Entreprises

### Contexte

Maintenant que vous avez les coordonnées GPS de chaque commune, vous allez interroger l’API Recherche Entreprises pour récupérer les entreprises situées dans un rayon de 5 km autour de chaque commune.

**Endpoint géographique :** `https://recherche-entreprises.api.gouv.fr/near_point`  
**Paramètres :** `lat`, `long`, `radius` (km), `per_page`, `page`

> ⚠️ **`/near_point` et non `/search`** : l’endpoint `/search` attend un paramètre `q` (nom d’entreprise). L’endpoint dédié à la recherche géographique est **`/near_point`**, il ne nécessite pas de `q`.

> ⚠️ **Rate limiting :** ajoutez `time.sleep(0.15)` entre chaque commune.


In [ ]:
# 🔧 TODO — Q13 : Explorer l’API Entreprises manuellement
#
# Faites une requête avec les coordonnées de Grenoble et un rayon de 5 km.
# Affichez le nombre total d’entreprises et les clés disponibles
# dans le premier résultat.
# Identifiez les champs : nom, SIREN, code NAF, effectifs, adresse.

result = safe_get('https://recherche-entreprises.api.gouv.fr/near_point', {
    ???   # TODO : paramètres
})

if result:
    print(f'Total entreprises : ???')   # TODO
    print(f'Clés disponibles : ???')    # TODO
    print(json.dumps(result['results'][0], indent=2, ensure_ascii=False))


In [1]:
# ✅ CODE FOURNI — Fonction de collecte des entreprises d’une commune
# /near_point est l’endpoint dédié à la recherche géographique.
# .get(’siege’, {}).get(’adresse’, ’’) : siege peut être None, le {} évite AttributeError.

def fetch_entreprises(nom_commune, lat, lon, radius=5):
    """Récupère TOUTES les entreprises via pagination automatique."""
    all_results = []
    page = 1

    while True:
        data = safe_get('https://recherche-entreprises.api.gouv.fr/near_point', {
            'lat':      lat,
            'long':     lon,
            'radius':   radius,
            'per_page': 25,
            'page':     page
        })

        if not data or not data.get('results'):
            break

        for e in data['results']:
            all_results.append({
                'nom_commune':      nom_commune,
                'nom':              e.get('nom_raison_sociale', ''),
                'siren':            e.get('siren', ''),
                'code_naf':         e.get('activite_principale', ''),
                'tranche_effectif': e.get('tranche_effectif_salarie', ''),
                'adresse':          e.get('siege', {}).get('adresse', ''),
                'cp_entreprise':    e.get('siege', {}).get('code_postal', ''),
            })

        # Vérifier s'il reste des pages
        total = data.get('total_results', 0)
        if page * 25 >= total:
            break
        page += 1
        time.sleep(0.1)   # pause entre les pages de la même commune

    return all_results

test = fetch_entreprises('Grenoble', 45.188, 5.724)
print(f'Test Grenoble : {len(test)} entreprises')
if test: print(json.dumps(test[0], indent=2, ensure_ascii=False))


NameError: name 'safe_get' is not defined

In [ ]:
# 🔧 TODO — Q14, Q15 & Q16 : Collecter, sauvegarder, analyser
#
# Q14. Pour chaque commune de communes_enrichies.csv (statut OK ou MULTIPLE,
#      avec une latitude renseignée), appelez fetch_entreprises().
#      Pourquoi utiliser .extend() plutôt que .append() ?
#
# Q15. Sauvegardez l'ensemble dans entreprises_raw.csv.
#
# Q16. Combien d'entreprises au total ?
#      Quelles sont les 5 communes avec le plus d'entreprises ?

all_rows = []

with open('communes_enrichies.csv', encoding='utf-8') as f:
    communes = list(csv.DictReader(f))

print(f'{len(communes)} communes à traiter...')

for row in communes:
    if row['statut'] not in ('OK', 'MULTIPLE') or not row['latitude']:
        continue

    # TODO : appelez fetch_entreprises() avec les bons arguments
    entreprises = fetch_entreprises(???, ???, ???)

    # TODO : .extend() ou .append() ?
    all_rows.???(entreprises)

    print(f'{row["nom_commune"]:30s} : {len(entreprises)} entreprises')
    time.sleep(0.15)  # ← Rate limiting obligatoire !

print(f'\n✓ Total : {len(all_rows)} entreprises collectées')

# TODO : sauvegardez all_rows dans entreprises_raw.csv
if all_rows:
    with open('entreprises_raw.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=all_rows[0].keys())
        writer.writeheader()
        ???  # TODO
    print('✓ entreprises_raw.csv créé')

# TODO : top 5 communes (Counter)
par_commune = Counter(r['nom_commune'] for r in all_rows)
print('\nTop 5 communes :')
for commune, nb in ???:  # TODO
    print(f'  {commune:30s} : {nb}')


---
## Section 5 — Partie 3 : Nettoyage & consolidation

### Pourquoi nettoyer les données ?

Les données collectées depuis des APIs sont rarement parfaites. Dans notre cas :
- Une même entreprise peut apparaître dans plusieurs communes voisines → **doublons**
- Certains SIREN ne font pas 9 chiffres → **données invalides**
- Les noms de communes peuvent avoir des formats différents ("grenoble" vs "GRENOBLE") → **incohérences**
- Certains champs peuvent être vides → **valeurs manquantes**

La règle d'or : **Garbage In, Garbage Out**. Des données sales produisent des analyses fausses.

### Exercice 5.1 — Audit de qualité

In [ ]:
# 🔧 TODO — Q17 & Q18 : Audit qualité du dataset brut
#
# Q17. Chargez entreprises_raw.csv avec pandas.
#      Affichez : nombre de lignes, nombre de doublons exacts,
#      valeurs manquantes par colonne.
#
# Q18. Vérifiez le format des SIREN : ils doivent tous faire exactement
#      9 chiffres. Combien sont invalides ?

df = pd.read_csv('entreprises_raw.csv')

print('=== AUDIT QUALITÉ ===')
print(f'Lignes     : ???')   # TODO
print(f'Colonnes   : ???')   # TODO
print(f'Doublons   : ???')   # TODO
print(f'\nValeurs manquantes :')
print(???)                   # TODO
print(f'\nExemples de SIREN : ???')  # TODO : 5 exemples


In [ ]:
# ✅ CODE FOURNI — Nettoyage complet avec explications
# Chaque transformation est commentée pour expliquer le 'pourquoi'

df_clean = df.drop_duplicates()  # Supprime les lignes 100% identiques
print(f'Après suppression des doublons : {len(df_clean)} lignes (-{len(df)-len(df_clean)})')

# Valider le format SIREN : exactement 9 chiffres
# astype(str).str.strip() : convertit en texte et enlève les espaces parasites
df_clean['siren'] = df_clean['siren'].astype(str).str.strip()
mask_invalide = ~df_clean['siren'].str.match(r'^\d{9}$')  # ^ = début, \d = chiffre, {9} = exactement 9, $ = fin
print(f'SIREN invalides supprimés : {mask_invalide.sum()}')
df_clean = df_clean[~mask_invalide]  # On supprime les lignes avec SIREN invalide

# Normaliser les noms de communes
# CRITIQUE pour la jointure ensuite : 'Grenoble' ≠ 'GRENOBLE' pour Python
df_clean['nom_commune'] = df_clean['nom_commune'].str.strip().str.upper()

# Remplir les colonnes texte vides par chaîne vide
# On préfère '' (vide) à 'nan' ou NaN pour les colonnes texte
for col in ['nom', 'code_naf', 'adresse']:
    df_clean[col] = df_clean[col].fillna('')

print(f'Dataset nettoyé : {len(df_clean)} lignes')

In [ ]:
# 🔧 TODO — Q19 & Q20 : Nettoyage et rapport de qualité
#
# Q19. Le nettoyage ci-dessus est fourni.
#      Expliquez en commentaire le rôle de chaque étape :
#      - Pourquoi .str.upper() sur nom_commune ?
#      - Pourquoi supprimer les lignes avec SIREN invalide plutôt que de les garder ?
#      - Pourquoi fillna('') pour les colonnes texte ?
#
# Q20. Créez le rapport de qualité (rapport_qualite.json)
#      avec : total_initial, total_final, doublons_supprimes, siren_supprimes.

# TODO : répondez en commentaire (Q19)
# .str.upper() : ???
# Suppression SIREN invalides : ???
# fillna('') : ???

# TODO : créez le rapport (Q20)
rapport = {
    'total_initial':      ???,
    'total_final':        ???,
    'doublons_supprimes': ???,
    'siren_supprimes':    ???,  # nombre de lignes supprimées pour SIREN invalide
}

with open('rapport_qualite.json', 'w') as f:
    json.dump(rapport, f, indent=2)

print('Rapport de qualité :')
for k, v in rapport.items():
    print(f'  {k:25s} : {v}')
print('✓ rapport_qualite.json créé')

### Exercice 5.3 — Fusion des datasets (merge)

Vous avez deux fichiers :
- `df_clean` : les entreprises avec leurs communes (mais sans coordonnées GPS)
- `communes_enrichies.csv` : les communes avec leurs coordonnées GPS

Vous voulez ajouter les colonnes `latitude` et `longitude` à chaque entreprise. Pour cela, on utilise `merge()` — c'est l'équivalent d'un **JOIN SQL** en pandas.

Un `merge() LEFT` signifie : _je garde TOUTES les lignes de gauche (entreprises), et j'y ajoute les infos de droite (coordonnées) quand elles existent_. Si une commune n'a pas de coordonnées, l'entreprise reste dans le dataset avec latitude=NaN.

**Point critique :** La clé de jointure (`nom_commune`) doit être formatée **identiquement** dans les deux DataFrames. `'Grenoble'` et `'GRENOBLE'` ne se rejoindront pas !

In [ ]:
# 🔧 TODO — Q21, Q22, Q23 & Q24 : Fusion et export final
#
# Q21. Avant de fusionner, normalisez nom_commune dans df_communes.
#      Pourquoi est-ce indispensable ?
#
# Q22. Réalisez un merge() LEFT entre df_clean et les colonnes
#      latitude/longitude de df_communes.
#      Qu'est-ce qu'un merge LEFT (par opposition à INNER) ?
#
# Q23. Combien d'entreprises n'ont pas de coordonnées GPS après la fusion ?
#      Comment l'expliquez-vous ?
#
# Q24. Exportez le résultat final dans dataset_final_kit1.csv.

df_communes = pd.read_csv('communes_enrichies.csv')

# TODO Q21 : normaliser nom_commune dans df_communes
df_communes['nom_commune'] = ???

# TODO Q22 : merge LEFT
df_final = df_clean.merge(
    df_communes[['nom_commune', 'latitude', 'longitude']],
    on=???,
    how=???
)

# TODO Q23 : compter les entreprises sans coordonnées
sans_coords = ???
print(f'Entreprises sans coordonnées GPS : {sans_coords}')

# TODO Q24 : exporter
???
print(f'✓ dataset_final_kit1.csv : {len(df_final)} lignes, {len(df_final.columns)} colonnes')


---
## ✅ Bilan du Kit 1

Si toutes les cellules se sont exécutées sans erreur, vous avez produit :

| Fichier | Description |
|---|---|
| `images_filtered.json` | Liste d'images Picsum filtrées par prénom d'auteur |
| `communes_enrichies.csv` | 49 communes avec coordonnées GPS et codes postaux |
| `entreprises_raw.csv` | Entreprises brutes collectées |
| `rapport_qualite.json` | Statistiques de nettoyage |
| `dataset_final_kit1.csv` | Dataset consolidé, prêt pour le Kit 2 |

**Ce dataset sera votre point de départ pour le Kit 2 — modélisation et pipeline ETL.**